In [2]:
# Install dependencies
# If you are on Colab and %pip causes issues, replace it with !pip

%pip install robustbench torchsde einops --quiet
%pip install -U pip wheel ninja --quiet
%pip install git+https://github.com/fra31/auto-attack
!python -m pip install --no-cache-dir "setuptools<81" --quiet

import os, sys

if not os.path.exists("DiffPure"):
    !git clone https://github.com/NVlabs/DiffPure.git

sys.path.insert(0, "DiffPure")
sys.path.insert(0, "DiffPure/score_sde")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 23.1 MB/s eta 0:00:00
  Cloning https://github.com/fra31/auto-attack to /tmp/pip-req-build-3hz5k0a0
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack /tmp/pip-req-build-3hz5k0a0
  Resolved https://github.com/fra31/auto-attack to commit a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for autoattack: filename=autoattack-0.1-py3-none-any.whl size=36303 sha256=6635565213292ba1ae570f849815da00bdf303abe9eaf2b2256c63a3f771c092
  Stored in directory: /tmp/pip-ephem-wheel-cache-ocpzco_u/wheels/5a/06/2e/e5e2d58dcb2d67ed9e5dbbd7752368f6c68c97cd3f629ba1b4
Successfully built autoattack
Cloning into 'DiffPure'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% 

## Checkpoint path

Set `CKPT_PATH` to your CIFAR-10 score-SDE checkpoint.  
If you are in Colab and the checkpoint is in Drive, you can mount Drive below.

In [3]:
# Optional Colab Drive mount
# Uncomment if needed

from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH
CKPT_PATH = '/content/drive/MyDrive/checkpoint_8.pth'

assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
print("Using checkpoint:", CKPT_PATH)

Mounted at /content/drive
Using checkpoint: /content/drive/MyDrive/checkpoint_8.pth


In [4]:
import os
import io
import time
import math
import random
import argparse
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import torchsde

from robustbench.data import load_cifar10
from robustbench.utils import load_model as rb_load_model

from score_sde.losses import get_optimizer
from score_sde.models import utils as mutils
from score_sde.models.ema import ExponentialMovingAverage
from score_sde import sde_lib
import score_sde.models.ncsnpp
import pandas as pd


In [5]:
# -----------------------------
# Config
# -----------------------------
SEED = 42
NUM_EXAMPLES = 20
START_INDEX = 0

JPEG_QUALITY = 75

# PGD Linf attack settings
ATTACK_EPS = 8.0 / 255.0
ATTACK_ALPHA = 2.0 / 255.0
ATTACK_STEPS = 10

# DiffPure settings
SCORE_TYPE = "score_sde"
SAMPLE_STEP = 1
USE_BM = False

# Requested visualization times
DP_START_T = 0.075
JDP_START_T = 0.050   # If you intended 0.05 instead, change only this line.

# Load enough CIFAR-10 samples for the requested slice
N_SAMPLES = START_INDEX + NUM_EXAMPLES

# Output root
if Path('/content/drive/MyDrive').exists():
    OUT_ROOT = Path('/content/drive/MyDrive/diffpure_ablation_examples_start_only')
else:
    OUT_ROOT = Path('./diffpure_ablation_examples_start_only')

OUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)
print("Saving outputs to:", OUT_ROOT.resolve())

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_seed()


Device: cuda
Saving outputs to: /content/drive/MyDrive/diffpure_ablation_examples_start_only


## Load CIFAR-10 subset

In [6]:
x_test, y_test = load_cifar10(n_examples=N_SAMPLES, data_dir="./data")
x_test, y_test = x_test.to(DEVICE), y_test.to(DEVICE)

x_subset = x_test[START_INDEX:START_INDEX + NUM_EXAMPLES]
y_subset = y_test[START_INDEX:START_INDEX + NUM_EXAMPLES]

print("subset shape:", x_subset.shape)
print("labels:", y_subset.detach().cpu().tolist())


100%|██████████| 170M/170M [00:01<00:00, 92.0MB/s]


subset shape: torch.Size([20, 3, 32, 32])
labels: [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6]


## Load the score-SDE model and the CIFAR-10 Linf classifier

In [7]:
def build_config(device):
    cfg = argparse.Namespace()
    cfg.data = argparse.Namespace()
    cfg.data.dataset = "CIFAR10"
    cfg.data.image_size = 32
    cfg.data.num_channels = 3
    cfg.data.centered = True

    cfg.model = argparse.Namespace()
    cfg.model.name = "ncsnpp"
    cfg.model.scale_by_sigma = False
    cfg.model.ema_rate = 0.9999
    cfg.model.normalization = "GroupNorm"
    cfg.model.nonlinearity = "swish"
    cfg.model.nf = 128
    cfg.model.ch_mult = (1, 2, 2, 2)
    cfg.model.num_res_blocks = 8
    cfg.model.attn_resolutions = (16,)
    cfg.model.resamp_with_conv = True
    cfg.model.conditional = True
    cfg.model.fir = False
    cfg.model.fir_kernel = [1, 3, 3, 1]
    cfg.model.skip_rescale = True
    cfg.model.resblock_type = "biggan"
    cfg.model.progressive = "none"
    cfg.model.progressive_input = "none"
    cfg.model.progressive_combine = "sum"
    cfg.model.attention_type = "ddpm"
    cfg.model.init_scale = 0.0
    cfg.model.fourier_scale = 16
    cfg.model.conv_size = 3
    cfg.model.sigma_min = 0.01
    cfg.model.sigma_max = 50
    cfg.model.num_scales = 1000
    cfg.model.beta_min = 0.1
    cfg.model.beta_max = 20.0
    cfg.model.dropout = 0.1
    cfg.model.embedding_type = "positional"

    cfg.optim = argparse.Namespace()
    cfg.optim.optimizer = "Adam"
    cfg.optim.lr = 2e-4
    cfg.optim.beta1 = 0.9
    cfg.optim.eps = 1e-8
    cfg.optim.weight_decay = 0.0
    cfg.optim.warmup = 5000
    cfg.optim.grad_clip = 1.0

    cfg.device = device
    cfg.seed = SEED
    return cfg


def restore_checkpoint(ckpt_path, state, device):
    loaded = torch.load(ckpt_path, map_location=device)
    state["optimizer"].load_state_dict(loaded["optimizer"])
    state["model"].load_state_dict(loaded["model"], strict=False)
    state["ema"].load_state_dict(loaded["ema"])
    state["step"] = loaded["step"]


def load_score_sde_model(ckpt_path, device):
    config = build_config(device)
    model = mutils.create_model(config)
    optimizer = get_optimizer(config, model.parameters())
    ema = ExponentialMovingAverage(model.parameters(), decay=config.model.ema_rate)
    state = dict(step=0, optimizer=optimizer, model=model, ema=ema)
    restore_checkpoint(ckpt_path, state, device)
    ema.copy_to(model.parameters())
    model.eval().to(device)
    print(f"Loaded score_sde model (step={state['step']})")
    return model


score_model = load_score_sde_model(CKPT_PATH, DEVICE)

classifier_inf = rb_load_model(
    model_name="Standard",
    dataset="cifar10",
    threat_model="Linf"
).eval().to(DEVICE)

print("Loaded score model and CIFAR-10 Linf classifier.")

Loaded score_sde model (step=400005)


Downloading...
From (original): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC
From (redirected): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC&confirm=t&uuid=21ba8266-c2ec-4c3a-ba11-5417bb9291ff
To: /content/models/cifar10/Linf/Standard.pt
100%|██████████| 292M/292M [00:01<00:00, 149MB/s]


Loaded score model and CIFAR-10 Linf classifier.


In [8]:
class RevVPSDE(nn.Module):
    def __init__(self, model, score_type="score_sde",
                 beta_min=0.1, beta_max=20.0, N=1000, img_shape=(3, 32, 32)):
        super().__init__()
        self.model = model
        self.score_type = score_type
        self.img_shape = img_shape
        self.beta_0 = beta_min
        self.beta_1 = beta_max
        self.N = N
        self.discrete_betas = torch.linspace(beta_min / N, beta_max / N, N)
        self.alphas = 1.0 - self.discrete_betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.vpsde = sde_lib.VPSDE(beta_min=beta_min, beta_max=beta_max, N=N)
        self.noise_type = "diagonal"
        self.sde_type = "ito"

    def vpsde_fn(self, t, x):
        beta_t = self.beta_0 + t * (self.beta_1 - self.beta_0)
        return -0.5 * beta_t[:, None] * x, torch.sqrt(beta_t)

    def rvpsde_fn(self, t, x, return_type="drift"):
        drift, diffusion = self.vpsde_fn(t, x)
        if return_type == "drift":
            x_img = x.view(-1, *self.img_shape)
            score_fn = mutils.get_score_fn(
                self.vpsde, self.model, train=False, continuous=True
            )
            score = score_fn(x_img, t).view(x.shape[0], -1)
            return drift - diffusion[:, None] ** 2 * score
        return diffusion

    def f(self, t, x):
        return -self.rvpsde_fn(1 - t.expand(x.shape[0]), x, "drift")

    def g(self, t, x):
        d = self.rvpsde_fn(1 - t.expand(x.shape[0]), x, "diffusion")
        return d[:, None].expand(x.shape)


rev_vpsde = RevVPSDE(score_model, SCORE_TYPE).to(DEVICE)
alphas_cumprod = rev_vpsde.alphas_cumprod.to(DEVICE)

print("RevVPSDE ready.")

RevVPSDE ready.


## JPEG helper

In [9]:
def jpeg_compress_tensor(x, quality=75):
    """True JPEG compression. x in [0,1] -> [0,1]."""
    device = x.device
    out = []
    for img in x:
        arr = (img.detach().cpu().clamp(0, 1) * 255).byte().permute(1, 2, 0).numpy()
        buf = io.BytesIO()
        Image.fromarray(arr).save(buf, format="JPEG", quality=quality)
        buf.seek(0)
        out.append(TF.to_tensor(Image.open(buf).convert("RGB")).to(device))
    return torch.stack(out)

print("JPEG helper ready.")

JPEG helper ready.


## DiffPure helpers

We separate:
- **forward noising**: create the noisy state at a chosen time
- **reverse diffusion**: go from that noisy state back to `t=0`

In [10]:
def time_to_steps(t_float, N=1000):
    """Convert e.g. 0.075 -> 75 steps when N=1000."""
    return max(1, int(round(t_float * N)))


def forward_noise_from_t(x_in, alphas_cumprod, t_float, device):
    """Create the noisy image used as the starting point for reverse diffusion."""
    t_steps = time_to_steps(t_float, N=len(alphas_cumprod))
    x0 = (x_in * 2.0 - 1.0).to(device)
    e = torch.randn_like(x0)
    a = alphas_cumprod[t_steps - 1]
    x_noisy = x0 * a.sqrt() + e * (1.0 - a).sqrt()
    x_noisy_01 = ((x_noisy + 1.0) / 2.0).clamp(0.0, 1.0)
    return x_noisy, x_noisy_01, t_steps


def reverse_segment(x_state, rev_vpsde, tau_start, tau_end, device, bm=None):
    """Run one reverse-diffusion segment from tau_start to tau_end.

    Here tau = 1 - t, so larger tau means later in the reverse process.
    x_state is in the centered [-1, 1] space expected by the reverse SDE.
    """
    B, C, H, W = x_state.shape
    ts = torch.tensor([tau_start, tau_end], device=device)
    x_flat = x_state.view(B, -1)
    if bm is not None:
        xs = torchsde.sdeint_adjoint(rev_vpsde, x_flat, ts, method="euler", bm=bm)
    else:
        xs = torchsde.sdeint_adjoint(rev_vpsde, x_flat, ts, method="euler")
    return xs[-1].view(B, C, H, W)


def run_diffpure_trajectory(x_in, start_t, capture_ts, rev_vpsde, alphas_cumprod, device, use_bm=True):
    """Run the correct DiffPure pipeline once and capture intermediate snapshots.

    Important: this is ONE trajectory per row. We first noise the input once at
    start_t, then continue the reverse process and capture states at the requested
    times. We do not re-run DiffPure independently for each displayed t, and we do
    not re-noise the intermediate outputs.

    Args:
        x_in: input image in [0, 1] of shape (B, C, H, W)
        start_t: starting diffusion level, e.g. 0.075
        capture_ts: list like [0.075, 0.050, 0.025, 0.0]

    Returns:
        snapshots: dict mapping each capture t to the corresponding image in [0, 1]
    """
    capture_ts = list(capture_ts)
    capture_ts = sorted(capture_ts, reverse=True)
    assert abs(capture_ts[0] - start_t) < 1e-8, "capture_ts must start with start_t"

    x_noisy, x_noisy_01, t_steps = forward_noise_from_t(x_in, alphas_cumprod, start_t, device)
    snapshots = {start_t: x_noisy_01}

    # tau is the forward variable used by RevVPSDE integration.
    tau_start = 1.0 - start_t
    tau_final = 1.0 - 1e-5

    bm = None
    if use_bm:
        B, C, H, W = x_noisy.shape
        bm = torchsde.BrownianInterval(
            t0=tau_start, t1=tau_final, size=(B, C * H * W), device=device
        )

    current = x_noisy
    prev_tau = tau_start

    for t_next in capture_ts[1:]:
        next_tau = tau_final if t_next <= 0.0 else 1.0 - t_next
        current = reverse_segment(current, rev_vpsde, prev_tau, next_tau, device, bm=bm)
        snapshots[t_next] = ((current + 1.0) / 2.0).clamp(0.0, 1.0)
        prev_tau = next_tau

    return snapshots


## Simple PGD $\ell_\infty$ attack for visualization

In [11]:
def pgd_linf_attack(model, x, y, epsilon=8/255, alpha=2/255, steps=10):
    model.eval()

    x_adv = x.clone().detach()
    x_adv = x_adv + torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
    x_adv = torch.clamp(x_adv, 0.0, 1.0)

    for _ in range(steps):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]

        x_adv = x_adv.detach() + alpha * grad.sign()
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
        x_adv = torch.clamp(x_adv, 0.0, 1.0)

    return x_adv.detach()

print("PGD Linf attack ready.")

PGD Linf attack ready.


## Utilities for predictions, saving, plotting, and summaries

In [12]:
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def predict_with_conf(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)
    return int(pred.item()), float(conf.item())

def denorm_np(x):
    x = x.detach().cpu().clamp(0, 1)[0]
    return x.permute(1, 2, 0).numpy()

def save_tensor_image(x, path):
    arr = (denorm_np(x) * 255).astype(np.uint8)
    Image.fromarray(arr).save(path)

def make_title(title, model, x):
    pred, conf = predict_with_conf(model, x)
    return f"{title}\n{CIFAR10_CLASSES[pred]} ({conf*100:.1f}%)"

def make_clean_ref_title(true_label):
    """Clean reference uses the original/ground-truth label only, with no confidence."""
    return f"Clean Ref\n{CIFAR10_CLASSES[int(true_label)]}"

def float_tag(t):
    return str(t).replace('.', '_')

def case_type(dp_correct, jdp_correct):
    if dp_correct and jdp_correct:
        return "both_good"
    if dp_correct and not jdp_correct:
        return "dp_only_good"
    if not dp_correct and jdp_correct:
        return "jdp_only_good"
    return "both_fail"

def plot_row_save(images, titles, row_label=None, figsize=(12, 3.3), save_path=None, suptitle=None):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(denorm_np(img))
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    if row_label is not None:
        axes[0].set_ylabel(row_label, fontsize=12)

    if suptitle is not None:
        fig.suptitle(suptitle, fontsize=13)
        plt.tight_layout(rect=[0, 0, 1, 0.93])
    else:
        plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

def plot_two_rows_save(row1_images, row1_titles, row2_images, row2_titles,
                       row1_label="DiffPure", row2_label="J75+DP",
                       figsize=(12, 6.2), save_path=None, suptitle=None):
    cols = max(len(row1_images), len(row2_images))
    fig, axes = plt.subplots(2, cols, figsize=figsize)

    for r in range(2):
        for c in range(cols):
            axes[r, c].axis("off")

    for c, (img, title) in enumerate(zip(row1_images, row1_titles)):
        axes[0, c].imshow(denorm_np(img))
        axes[0, c].set_title(title, fontsize=10)
        axes[0, c].axis("off")

    for c, (img, title) in enumerate(zip(row2_images, row2_titles)):
        axes[1, c].imshow(denorm_np(img))
        axes[1, c].set_title(title, fontsize=10)
        axes[1, c].axis("off")

    axes[0, 0].set_ylabel(row1_label, fontsize=12)
    axes[1, 0].set_ylabel(row2_label, fontsize=12)

    if suptitle is not None:
        fig.suptitle(suptitle, fontsize=13)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
    else:
        plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


## Generate and save all examples

This cell loops over `NUM_EXAMPLES` examples and saves the outputs to Drive (or a local folder if Drive is not mounted).

For each example it saves:
- `dp_single.png`
- `j75_dp_single.png`
- `combined.png`
- raw stage images used in those figures
- `metadata.json`

It also writes a root-level `summary.csv` with the prediction outcomes, so you can quickly browse for good and failure cases.


In [ ]:
summary_rows = []

dp_tag = float_tag(DP_START_T)
jdp_tag = float_tag(JDP_START_T)

for local_idx in range(NUM_EXAMPLES):
    global_idx = START_INDEX + local_idx
    print(f"Processing example {local_idx + 1}/{NUM_EXAMPLES} (dataset index {global_idx})...")

    x_clean = x_subset[local_idx:local_idx + 1]
    y_clean = y_subset[local_idx:local_idx + 1]
    true_label = int(y_clean.item())
    true_name = CIFAR10_CLASSES[true_label]

    # Shared adversarial example
    set_seed(SEED + global_idx)
    x_adv = pgd_linf_attack(
        classifier_inf,
        x_clean,
        y_clean,
        epsilon=ATTACK_EPS,
        alpha=ATTACK_ALPHA,
        steps=ATTACK_STEPS,
    )

    # DiffPure only
    set_seed(SEED + 1000 + global_idx)
    dp_snaps = run_diffpure_trajectory(
        x_adv,
        start_t=DP_START_T,
        capture_ts=[DP_START_T, 0.0],
        rev_vpsde=rev_vpsde,
        alphas_cumprod=alphas_cumprod,
        device=DEVICE,
        use_bm=True,
    )
    dp_start = dp_snaps[DP_START_T]
    dp_final = dp_snaps[0.0]

    # JPEG-75 + DiffPure
    set_seed(SEED + 2000 + global_idx)
    x_adv_jpeg = jpeg_compress_tensor(x_adv, quality=JPEG_QUALITY)
    jdp_snaps = run_diffpure_trajectory(
        x_adv_jpeg,
        start_t=JDP_START_T,
        capture_ts=[JDP_START_T, 0.0],
        rev_vpsde=rev_vpsde,
        alphas_cumprod=alphas_cumprod,
        device=DEVICE,
        use_bm=True,
    )
    jdp_start = jdp_snaps[JDP_START_T]
    jdp_final = jdp_snaps[0.0]

    # Predictions
    clean_pred, clean_conf = predict_with_conf(classifier_inf, x_clean)
    adv_pred, adv_conf = predict_with_conf(classifier_inf, x_adv)
    dp_start_pred, dp_start_conf = predict_with_conf(classifier_inf, dp_start)
    dp_final_pred, dp_final_conf = predict_with_conf(classifier_inf, dp_final)
    jdp_start_pred, jdp_start_conf = predict_with_conf(classifier_inf, jdp_start)
    jdp_final_pred, jdp_final_conf = predict_with_conf(classifier_inf, jdp_final)

    clean_correct = int(clean_pred == true_label)
    adv_success = int(adv_pred != true_label)
    dp_correct = int(dp_final_pred == true_label)
    jdp_correct = int(jdp_final_pred == true_label)
    example_case = case_type(bool(dp_correct), bool(jdp_correct))

    case_dir = OUT_ROOT / example_case
    example_dir = case_dir / f"example_{global_idx:03d}"
    raw_dir = example_dir / "raw_images"
    example_dir.mkdir(parents=True, exist_ok=True)
    raw_dir.mkdir(parents=True, exist_ok=True)

    # Figure contents
    dp_images = [x_adv, dp_start, dp_final, x_clean]
    dp_titles = [
        make_title("Adversarial", classifier_inf, x_adv),
        make_title(f"DP(t={DP_START_T:.3f})", classifier_inf, dp_start),
        make_title("DP(t=0)", classifier_inf, dp_final),
        make_clean_ref_title(true_label),
    ]

    jdp_images = [x_adv, jdp_start, jdp_final, x_clean]
    jdp_titles = [
        make_title("Adversarial", classifier_inf, x_adv),
        make_title(f"J75+DP(t={JDP_START_T:.3f})", classifier_inf, jdp_start),
        make_title("J75+DP(t=0)", classifier_inf, jdp_final),
        make_clean_ref_title(true_label),
    ]

    # Save figures
    plot_row_save(
        dp_images,
        dp_titles,
        row_label="DiffPure",
        figsize=(12, 3.4),
        save_path=example_dir / "dp_single.png",
        suptitle=None,
    )

    plot_row_save(
        jdp_images,
        jdp_titles,
        row_label="J75+DP",
        figsize=(12, 3.4),
        save_path=example_dir / "j75_dp_single.png",
        suptitle=None,
    )

    plot_two_rows_save(
        dp_images,
        dp_titles,
        jdp_images,
        jdp_titles,
        row1_label="DiffPure",
        row2_label="J75+DP",
        figsize=(12, 6.2),
        save_path=example_dir / "combined.png",
        suptitle=None,
    )

    # Save raw images
    raw_save_map = {
        "clean_ref.png": x_clean,
        "adversarial.png": x_adv,
        f"dp_start_t_{dp_tag}.png": dp_start,
        "dp_final.png": dp_final,
        "jpeg75_from_adv.png": x_adv_jpeg,
        f"j75_dp_start_t_{jdp_tag}.png": jdp_start,
        "j75_dp_final.png": jdp_final,
    }

    for name, tensor in raw_save_map.items():
        save_tensor_image(tensor, raw_dir / name)

    metadata = {
        "global_index": global_idx,
        "true_label": true_name,
        "clean_pred": CIFAR10_CLASSES[clean_pred],
        "adv_pred": CIFAR10_CLASSES[adv_pred],
        "dp_start_pred": CIFAR10_CLASSES[dp_start_pred],
        "dp_final_pred": CIFAR10_CLASSES[dp_final_pred],
        "jdp_start_pred": CIFAR10_CLASSES[jdp_start_pred],
        "jdp_final_pred": CIFAR10_CLASSES[jdp_final_pred],
        "clean_conf": clean_conf,
        "adv_conf": adv_conf,
        "dp_start_conf": dp_start_conf,
        "dp_final_conf": dp_final_conf,
        "jdp_start_conf": jdp_start_conf,
        "jdp_final_conf": jdp_final_conf,
        "clean_correct": clean_correct,
        "adv_success": adv_success,
        "dp_correct": dp_correct,
        "jdp_correct": jdp_correct,
        "case_type": example_case,
        "attack_eps": ATTACK_EPS,
        "attack_alpha": ATTACK_ALPHA,
        "attack_steps": ATTACK_STEPS,
        "jpeg_quality": JPEG_QUALITY,
        "dp_start_t": DP_START_T,
        "jdp_start_t": JDP_START_T,
    }

    with open(example_dir / "metadata.json", "w") as f:
        import json
        json.dump(metadata, f, indent=2)

    summary_rows.append(metadata)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["case_type", "global_index"]).reset_index(drop=True)
summary_csv_path = OUT_ROOT / "summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("\nDone.")
print("Summary CSV saved to:", summary_csv_path)


Processing example 1/20 (dataset index 0)...


/usr/local/lib/python3.12/dist-packages/torchsde/_brownian/brownian_interval.py:599: UserWarning: Should have ta>=t0 but got ta=0.949999988079071 and t0=0.95.
  warnings.warn(f"Should have ta>=t0 but got ta={ta} and t0={self._start}.")


Processing example 2/20 (dataset index 1)...
Processing example 3/20 (dataset index 2)...
Processing example 4/20 (dataset index 3)...
Processing example 5/20 (dataset index 4)...
Processing example 6/20 (dataset index 5)...
Processing example 7/20 (dataset index 6)...
Processing example 8/20 (dataset index 7)...
Processing example 9/20 (dataset index 8)...
Processing example 10/20 (dataset index 9)...
Processing example 11/20 (dataset index 10)...
Processing example 12/20 (dataset index 11)...
Processing example 13/20 (dataset index 12)...
Processing example 14/20 (dataset index 13)...
Processing example 15/20 (dataset index 14)...
Processing example 16/20 (dataset index 15)...
Processing example 17/20 (dataset index 16)...
Processing example 18/20 (dataset index 17)...
Processing example 19/20 (dataset index 18)...
Processing example 20/20 (dataset index 19)...

Done.
Summary CSV saved to: /content/drive/MyDrive/diffpure_ablation_examples_start_only/summary.csv
